<a id="top"></a>

# Demo: the loop made visible (on a stub model)

```yaml
title: "Demo: the loop made visible (on a stub model)"
keywords: agent loop, run_loop, stub model, FakeModel, bind_tools, tool_calls, ToolMessage, natural termination, max_steps cap, tool failure, offline, teacher demo
estimated duration: 18
```

> **Lesson:** L10. Teacher demo — the loop concept from the [roadmap](../../../../docs/origin/lesson_roadmaps/L10/demos_or_activities.md). **No API key needed.** This demo drives the loop with a *scripted stub model* so termination, the `max_steps` cap, and tool-failure handling are **deterministic** — the same run every time. The live multi-step version is [L1006](L1006_lecture.ipynb).
>
> The point to land: **an agent is a loop, not a model.** The model is a stateless function call; the loop calls it, runs the tool it asked for, feeds the result back, and repeats — until the model stops asking (natural termination) or a cap you chose fires.

## Contents

- [1. Setup — tools, a stub model, and the loop](#1-setup--tools-a-stub-model-and-the-loop)
- [2. The loop itself](#2-the-loop-itself)
- [3. Natural termination — the happy path](#3-natural-termination--the-happy-path)
- [4. The max_steps cap — catching a runaway](#4-the-max_steps-cap--catching-a-runaway)
- [5. Tool failure as a message, not a crash](#5-tool-failure-as-a-message-not-a-crash)
- [6. Takeaways](#6-takeaways)

## 1. Setup — tools, a stub model, and the loop

Three pieces. (1) Two inline tools — `calculator` and `lookup` — and a **dispatch table** mapping each name to its function. (2) A scripted **`FakeModel`** (from the course's `common` layer) that mimics a LangChain chat model: `.bind_tools(...)` then `.invoke(messages)` returns the next scripted **`AIMessage`**, whose `.tool_calls` say which tools it wants run. (3) `dispatch`, which runs one tool and — **Rule 3** — turns any exception into an error **`ToolMessage`**. The loop can't tell the fake from a real `ChatAnthropic`; only the *script* is fixed.

In [1]:
from collections.abc import Callable
from dataclasses import dataclass

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.messages.tool import ToolCall

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {
    "tokyo": "37000000",
    "lagos": "15000000",
    "paris": "11000000",
}


def lookup(city: str) -> str:
    """Return a city's population from a tiny in-memory table, or raise KeyError if missing."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The dispatch table: tool NAME -> the Python function that runs it. The loop never
# hard-codes tool names; it looks each requested name up here to EXECUTE a call. The
# same functions, passed to model.bind_tools([...]), also ARE the schema a real model
# sees -- LangChain infers it from their type hints and docstrings.
TOOLS: dict[str, Callable[..., str]] = {"calculator": calculator, "lookup": lookup}

In [2]:
def dispatch(tools: dict[str, Callable[..., str]], call: ToolCall) -> ToolMessage:
    """Run one requested tool and return a ToolMessage carrying the result.

    On success: a ToolMessage (status="success") with the tool's output. On ANY
    failure (unknown tool name, or the tool raised): a ToolMessage with
    status="error" and a SHORT message (repr(exc)) -- never a traceback. The
    tool_call_id pairs the result back to the request; the loop keeps going and the
    model decides what to do next.
    """
    fn = tools.get(call["name"])
    if fn is None:
        return ToolMessage(
            content=f"no such tool {call['name']!r}",
            tool_call_id=call["id"],
            status="error",
        )
    try:
        output = fn(**call["args"])
        return ToolMessage(content=output, tool_call_id=call["id"], status="success")
    except Exception as exc:
        # repr(exc) is a class name + one line: e.g. KeyError('no population...').
        # Short, descriptive, cheap -- never dump a traceback at the model.
        return ToolMessage(content=repr(exc), tool_call_id=call["id"], status="error")

[↑ Back to top](#top)

## 2. The loop itself

`run_loop(model, tools, user_msg, max_steps)`. First `model.bind_tools(...)` so the model knows the tools; then each iteration: `.invoke(messages)` → an `AIMessage`. If it carried `tool_calls`, append that assistant turn, then run **every** call and append **one `ToolMessage` per call** (**Rule 1** — the message-history invariant), then loop; if it carried only text, return it. The `max_steps` cap forces a halt even if the model still wants tools.

In [3]:
@dataclass
class RunResult:
    """What the loop returns: the answer, how many model calls it took, and WHY it stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"


def run_loop(
    model: object,  # any bind_tools-capable chat model: FakeModel offline, ChatAnthropic live
    tools: dict[str, Callable[..., str]],
    user_msg: str,
    max_steps: int,
) -> RunResult:
    """Run a model -> tool -> model loop until the model stops asking for tools.

    Each iteration: invoke the tool-bound model; if the reply carried tool_calls, run
    EVERY one and append a ToolMessage per call, then loop; if it carried only text,
    return it (natural termination). The max_steps cap forces a halt even if the model
    still wants tools -- termination is a design decision.

    Tool failures are turned into messages, not crashes: a tool that raises becomes a
    ToolMessage with status="error", handed back so the model can decide what to do.
    """
    bound = model.bind_tools(list(tools.values()))
    messages: list[BaseMessage] = [HumanMessage(content=user_msg)]

    for step in range(1, max_steps + 1):
        reply = bound.invoke(messages)

        # No tool calls -> the model thinks it's done. NATURAL termination.
        if not reply.tool_calls:
            print(f"[step {step}] natural termination (no tool calls)")
            return RunResult(final_text=reply.text, iterations=step, termination="natural")

        # Rule 1: record the assistant turn, then answer EVERY tool call with a
        # matching ToolMessage before the next model call.
        messages.append(reply)
        for call in reply.tool_calls:
            print(f"[step {step}] tool call: {call['name']}({call['args']})")
            messages.append(dispatch(tools, call))

    # Fell out of the for-loop: the cap fired before the model finished.
    print(f"[stop] max_steps={max_steps} hit -- non-natural termination")
    return RunResult(final_text="", iterations=max_steps, termination="max_steps")

[↑ Back to top](#top)

## 3. Natural termination — the happy path

Script a model that calls `calculator`, then (seeing the result) calls `lookup`, then answers in plain text. Two tool-call turns, then a final message with **no `tool_calls`** — that is natural termination. Watch the per-step prints walk the loop.

In [4]:
happy_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "288 * 0 + 1"})),
    tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
    text_reply("Tokyo's population is about 37,000,000."),
]
model = FakeModel(happy_script)
result = run_loop(model, TOOLS, "What is the population of Tokyo?", max_steps=10)
print()
print("final_text :", result.final_text)
print("iterations :", result.iterations)
print("termination:", result.termination)

[step 1] tool call: calculator({'expression': '288 * 0 + 1'})
[step 2] tool call: lookup({'city': 'Tokyo'})
[step 3] natural termination (no tool calls)

final_text : Tokyo's population is about 37,000,000.
iterations : 3
termination: natural


[↑ Back to top](#top)

## 4. The max_steps cap — catching a runaway

Now a model that **never stops**: it keeps asking for the same `lookup` call. With a real model this is the classic runaway. The `FakeModel` reuses its last script line forever, so the loop would spin without a cap. `max_steps=4` catches it — and **hitting the cap is always a signal worth investigating**, not noise.

In [5]:
runaway_script = [
    tool_reply(tool_call("r1", "lookup", {"city": "Tokyo"})),
]
model = FakeModel(runaway_script)
result = run_loop(model, TOOLS, "Keep checking Tokyo's population.", max_steps=4)
print()
print("termination:", result.termination, "  <- the cap fired, not the model")
print("iterations :", result.iterations)
assert result.termination == "max_steps"

[step 1] tool call: lookup({'city': 'Tokyo'})
[step 2] tool call: lookup({'city': 'Tokyo'})
[step 3] tool call: lookup({'city': 'Tokyo'})
[step 4] tool call: lookup({'city': 'Tokyo'})
[stop] max_steps=4 hit -- non-natural termination

termination: max_steps   <- the cap fired, not the model
iterations : 4


[↑ Back to top](#top)

## 5. Tool failure as a message, not a crash

Script the model to look up a city **not in the table** (`lookup` raises `KeyError`), then recover by looking up one that *is*. Without `dispatch`'s `try/except` the `KeyError` would crash the whole agent. Instead the loop converts it to a `ToolMessage` with `status="error"`, the model 'sees' the error on its next turn, and finishes gracefully.

In [6]:
recover_script = [
    tool_reply(tool_call("f1", "lookup", {"city": "Atlantis"})),  # not in table -> KeyError
    tool_reply(tool_call("f2", "lookup", {"city": "Paris"})),  # recover
    text_reply("I couldn't find Atlantis, but Paris is about 11,000,000."),
]
model = FakeModel(recover_script)
result = run_loop(model, TOOLS, "Population of Atlantis? If unknown, try Paris.", max_steps=10)
print()
print("final_text :", result.final_text)
print("termination:", result.termination)
# Show the error ToolMessage the loop fed back after the first (failing) call:
bad_call = tool_call("f1", "lookup", {"city": "Atlantis"})
print("error ToolMessage handed to the model:", dispatch(TOOLS, bad_call))

[step 1] tool call: lookup({'city': 'Atlantis'})
[step 2] tool call: lookup({'city': 'Paris'})
[step 3] natural termination (no tool calls)

final_text : I couldn't find Atlantis, but Paris is about 11,000,000.
termination: natural
error ToolMessage handed to the model: content='KeyError("no population on file for \'Atlantis\'")' tool_call_id='f1' status='error'


[↑ Back to top](#top)

## 6. Takeaways

- **An agent is a loop, not a model.** The same `run_loop` drove all three runs above; only the *script* changed. (In [L1006](L1006_lecture.ipynb) only the *client* changes — a real model replaces the stub, loop unchanged.)
- **Termination is a design decision.** Natural termination = the model returned plain text (no `tool_calls`). The `max_steps` cap is a *safety net* for everything else; a loop with no cap is broken, not minimal.
- **Tool failures are messages, not exceptions.** `dispatch` turns a raise into a `ToolMessage(status="error")` with a short `repr(exc)` — never a traceback — and the model decides whether to retry (reinforces [L08](../L08/objectives.md)'s error thinking at the *loop* layer).
- **A stub model makes control flow testable.** Scripting the model is how you exercise termination and failure *offline and reproducibly* — the same mocking stance as the project's tests.
- Next: practice the loop in [L1004](L1004_lab_empty.ipynb), failures in [L1005](L1005_lab_empty.ipynb), then watch it drive a real model in [L1006](L1006_lecture.ipynb).

[↑ Back to top](#top)